In [1]:
import os
import torch
import numpy as np
from collections import defaultdict
import pprint
import pandas as pd
import sys
sys.path.append('../src')
from utils import round_to_perm


def summarize(values):
    return {
        'mean': np.nanmean(values),
        'sd': np.nanstd(values)
    }

B = 100  # Change as needed
n_i = 8 # Change as needed
result_folder = f'../data/results/B_{B}_n_{n_i}'

stats = {
    'GPmodel': defaultdict(list),
    'GPArealModel': defaultdict(list),
    'VIGP_unlinked': defaultdict(lambda: defaultdict(list))
}

for fname in os.listdir(result_folder):
    if not fname.startswith('results_seed_') or not fname.endswith('.pt'):
        continue
    path = os.path.join(result_folder, fname)
    result = torch.load(path, map_location='cpu')

    for model in ['GPmodel', 'GPArealModel']:
        if model in result:
            m = result[model]
            stats[model]['beta'].append(m.get('beta', [np.nan])[0])
            stats[model]['sigmasq'].append(m.get('sigmasq', np.nan))
            stats[model]['tausq'].append(m.get('tausq', np.nan))
            stats[model]['phi'].append(m.get('phi', np.nan))

    for tau in [0.1, 0.3, 0.5]:
        key = f'VIGP_unlinked_tau_{tau}'
        if key in result:
            vi = result[key]
            beta = vi.get('mu_lambda_beta', np.nan)
            sig_beta = vi.get('sigmasq_lambda_beta', np.nan)

            a1, b1 = vi.get('lambda_a1', np.nan), vi.get('lambda_b1', np.nan)
            a2, b2 = vi.get('lambda_a2', np.nan), vi.get('lambda_b2', np.nan)

            sigmasq = b1 / (a1 - 1) if a1 > 1 else np.nan
            tausq = b2 / (a2 - 1) if a2 > 1 else np.nan

            sd_sigmasq = (b1**2) / ((a1 - 1)**2 * (a1 - 2)) if a1 > 2 else np.nan
            sd_tausq = (b2**2) / ((a2 - 1)**2 * (a2 - 2)) if a2 > 2 else np.nan

            phi = vi.get('mean_phi', np.nan)

            stats['VIGP_unlinked'][tau]['beta'].append(beta)
            stats['VIGP_unlinked'][tau]['sigmasq'].append(sigmasq)
            stats['VIGP_unlinked'][tau]['tausq'].append(tausq)
            stats['VIGP_unlinked'][tau]['phi'].append(phi)
            stats['VIGP_unlinked'][tau]['sd_sigmasq'].append(sd_sigmasq)
            stats['VIGP_unlinked'][tau]['sd_tausq'].append(sd_tausq)
            stats['VIGP_unlinked'][tau]['sd_beta'].append(np.sqrt(sig_beta))

summary = {
    model: {k: summarize(v) for k, v in stats[model].items()}
    for model in ['GPmodel', 'GPArealModel']
}
summary['VIGP_unlinked'] = {
    tau: {
        'beta': {'mean': np.nanmean(v['beta']), 'sd': np.nanmean(v['sd_beta'])},
        'sigmasq': {'mean': np.nanmean(v['sigmasq']), 'sd': np.nanmean(v['sd_sigmasq'])},
        'tausq': {'mean': np.nanmean(v['tausq']), 'sd': np.nanmean(v['sd_tausq'])},
        'phi': {'mean': np.nanmean(v['phi']), 'sd': np.nanstd(v['phi'])}
    } for tau, v in stats['VIGP_unlinked'].items()
}

# Prepare data for the table
data = {
    'GPmodel': [summary['GPmodel']['beta']['mean'], summary['GPmodel']['beta']['sd']],
    'GPArealModel': [summary['GPArealModel']['beta']['mean'], summary['GPArealModel']['beta']['sd']],
}

for tau in [0.1, 0.3, 0.5]:
    data[f'VI_tau_{tau}'] = [
        summary['VIGP_unlinked'][tau]['beta']['mean'],
        summary['VIGP_unlinked'][tau]['beta']['sd']
    ]

# Create a DataFrame for the table
table = pd.DataFrame(data, index=['Mean', 'SD'])

# Add rows for beta, phi, tausq, and sigmasq in "mean (sd)" format
data_extended = {
    'GPmodel': [
        f"{summary['GPmodel']['beta']['mean']:.4f} ({summary['GPmodel']['beta']['sd']:.4f})",
        f"{summary['GPmodel']['phi']['mean']:.4f} ({summary['GPmodel']['phi']['sd']:.4f})",
        f"{summary['GPmodel']['tausq']['mean']:.4f} ({summary['GPmodel']['tausq']['sd']:.4f})",
        f"{summary['GPmodel']['sigmasq']['mean']:.4f} ({summary['GPmodel']['sigmasq']['sd']:.4f})"
    ],
    'GPArealModel': [
        f"{summary['GPArealModel']['beta']['mean']:.4f} ({summary['GPArealModel']['beta']['sd']:.4f})",
        f"{summary['GPArealModel']['phi']['mean']:.4f} ({summary['GPArealModel']['phi']['sd']:.4f})",
        f"{summary['GPArealModel']['tausq']['mean']:.4f} ({summary['GPArealModel']['tausq']['sd']:.4f})",
        f"{summary['GPArealModel']['sigmasq']['mean']:.4f} ({summary['GPArealModel']['sigmasq']['sd']:.4f})"
    ],
}

for tau in [0.1, 0.3, 0.5]:
    data_extended[f'VI_tau_{tau}'] = [
        f"{summary['VIGP_unlinked'][tau]['beta']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['beta']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['phi']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['phi']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['tausq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['tausq']['sd']:.4f})",
        f"{summary['VIGP_unlinked'][tau]['sigmasq']['mean']:.4f} ({summary['VIGP_unlinked'][tau]['sigmasq']['sd']:.4f})"
    ]

# Create a DataFrame for the extended table
table_extended = pd.DataFrame(
    data_extended,
    index=['Beta', 'Phi', 'Tausq', 'Sigmasq']
)

# Display the tables
print(table)
print(table_extended)


       GPmodel  GPArealModel  VI_tau_0.1  VI_tau_0.3  VI_tau_0.5
Mean  7.947275      7.423395    2.651712    7.980823    7.401990
SD    0.115043      0.610162    0.152254    0.062088    0.070766
                 GPmodel     GPArealModel       VI_tau_0.1       VI_tau_0.3  \
Beta     7.9473 (0.1150)  7.4234 (0.6102)  2.6517 (0.1523)  7.9808 (0.0621)   
Phi      5.2495 (0.9624)  4.4650 (1.5397)  4.7303 (0.0006)  4.7285 (0.0059)   
Tausq    0.2499 (0.0473)  1.1599 (0.5948)  7.4072 (0.1385)  1.0237 (0.0026)   
Sigmasq  3.9640 (0.5037)  4.0480 (0.7208)  1.7260 (0.0084)  2.2756 (0.0134)   

              VI_tau_0.5  
Beta     7.4020 (0.0708)  
Phi      4.7301 (0.0012)  
Tausq    1.3644 (0.0047)  
Sigmasq  2.4167 (0.0151)  


/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_7900/586168993.py:32: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  result = torch.load(path, map_location='cpu')


In [2]:
print(f"B = {B}, and n = {n_i}.")
print(table_extended)

B = 100, and n = 8.
                 GPmodel     GPArealModel       VI_tau_0.1       VI_tau_0.3  \
Beta     7.9473 (0.1150)  7.4234 (0.6102)  2.6517 (0.1523)  7.9808 (0.0621)   
Phi      5.2495 (0.9624)  4.4650 (1.5397)  4.7303 (0.0006)  4.7285 (0.0059)   
Tausq    0.2499 (0.0473)  1.1599 (0.5948)  7.4072 (0.1385)  1.0237 (0.0026)   
Sigmasq  3.9640 (0.5037)  4.0480 (0.7208)  1.7260 (0.0084)  2.2756 (0.0134)   

              VI_tau_0.5  
Beta     7.4020 (0.0708)  
Phi      4.7301 (0.0012)  
Tausq    1.3644 (0.0047)  
Sigmasq  2.4167 (0.0151)  


In [3]:
torch.norm(result['VIGP_unlinked_tau_0.3']['V_X_star'])

tensor(2.8097)

In [20]:
torch.sqrt(torch.tensor(8))

tensor(2.8284)

In [4]:
B = 100  # Change as needed
n_i = 8 # Change as needed
result_folder = f'../data/B_{B}_n_{n_i}'
seed = 11  # Example seed
file_path = os.path.join(result_folder, f'data_seed_{seed}.pt')
data = torch.load(file_path, map_location='cpu')

# Extract perm_matrix_x and perm_matrix_s
perm_matrix_x = data['perm_matrix_x']
perm_matrix_s = data['perm_matrix_s']

print("perm_matrix_x:", perm_matrix_x)
print("perm_matrix_s:", perm_matrix_s)

perm_matrix_x: tensor([[0., 0., 0., 0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.]])
perm_matrix_s: tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 1., 0.]])


/var/folders/w7/jxz2zn316391355qstwl03940000gn/T/ipykernel_7900/2669334214.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(file_path, map_location='cpu

In [8]:
torch.sum(perm_matrix_x.T @ round_to_perm(result['VIGP_unlinked_tau_0.5']['M_X_star'].detach().numpy()))

tensor(8., dtype=torch.float64)

In [14]:
torch.sum(perm_matrix_s.T @ round_to_perm(result['VIGP_unlinked_tau_0.3']['M_S_star'].detach().numpy()))

tensor(8., dtype=torch.float64)